In [7]:
import os
import re
from typing import List

# New imports for OCR and CSV
import pytesseract
from pdf2image import convert_from_path
#from langchain.document_loaders import PyPDFLoader, DirectoryLoader, CSVLoader
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, CSVLoader

#from langchain.docstore.document import Document
from langchain_core.documents import Document

#from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

#from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

#from langchain_pinecone import PineconeVectorStore
from langchain_community.vectorstores import Pinecone

from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import time

# --- Configuration ---
load_dotenv()
PINECONE_API_KEY_TA = os.getenv("PINECONE_API_KEY_TA")
if not PINECONE_API_KEY_TA:
    raise ValueError("PINECONE_API_KEY not found in .env file.")

# Paths to your data folders
PATH_EN_PDFS = r"D:\CyChat\CyChat2\data\d-en"
PATH_TA_PDFS = r"D:\CyChat\CyChat2\data\d-ta" # Assuming this is your Tamil PDF path
PATH_CSV = r"D:\CyChat\CyChat2\data\csv"     # Assuming this is your CSV path

# CRITICAL: Use a multilingual model for ALL documents
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_DIMENSION = 384
INDEX_NAME = "cyberchat" # New index name



In [8]:
# --- Data Loading Functions ---

def load_english_pdfs(path: str) -> List[Document]:
    """Loads text-based PDFs and adds 'lang' metadata."""
    print(f"Loading English PDFs from {path}...")
    loader = DirectoryLoader(
        path,
        glob="*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True
    )
    docs = loader.load()
    for doc in docs:
        doc.metadata["lang"] = "en" # Add language metadata
        doc.metadata["source_type"] = "pdf"
    return docs

def load_tamil_ocr_pdfs(path: str) -> List[Document]:
    """
    Loads image-based Tamil PDFs using Tesseract OCR.
    """
    print(f"Loading Tamil (OCR) PDFs from {path}...")
    docs = []
    for filename in os.listdir(path):
        if not filename.lower().endswith(".pdf"):
            continue
        
        file_path = os.path.join(path, filename)
        print(f"  Processing {filename}...")
        try:
            # Convert PDF pages to images
            images = convert_from_path(file_path)
            
            for i, image in enumerate(images):
                # Use pytesseract to extract Tamil text
                # You MUST have the 'tam' (Tamil) language pack installed for Tesseract
                text = pytesseract.image_to_string(image, lang='tam')
                
                if text.strip(): # Only add if text was found
                    metadata = {
                        "source": file_path,
                        "page": i + 1,
                        "lang": "ta", # Add language metadata
                        "source_type": "pdf_ocr"
                    }
                    docs.append(Document(page_content=text, metadata=metadata))
        except Exception as e:
            print(f"    Failed to process {filename} on page {i+1}: {e}")
            
    print(f"Loaded {len(docs)} pages from Tamil PDFs.")
    return docs

def load_csv_files(path: str) -> List[Document]:
    """Loads CSV files and adds 'lang' metadata."""
    print(f"Loading CSVs from {path}...")
    loader = DirectoryLoader(
        path,
        glob="*.csv",
        loader_cls=CSVLoader, # Use CSVLoader
        show_progress=True
    )
    docs = loader.load()
    for doc in docs:
        doc.metadata["lang"] = "en" # Assume CSV data is English (or change as needed)
        doc.metadata["source_type"] = "csv"
    return docs

def clean_document_content(docs: List[Document]) -> List[Document]:
    """Basic cleaning for all loaded documents."""
    print("Cleaning all documents...")
    cleaned_docs = []
    for doc in docs:
        text = doc.page_content
        text = re.sub(r'\n{3,}', '\n\n', text) # Consolidate blank lines
        text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text) # Fix hyphenation
        text = re.sub(r'\s{2,}', ' ', text) # Consolidate whitespace
        
        if text.strip(): # Only keep if content remains
            doc.page_content = text
            cleaned_docs.append(doc)
    return cleaned_docs

def split_docs(documents: List[Document]) -> List[Document]:
    """Split documents into chunks."""
    print("Splitting documents...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50, # Increased overlap for better context
        length_function=len
    )
    texts_chunk = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(texts_chunk)}")
    return texts_chunk



In [ ]:


# --- Main Ingestion Logic ---
def main():
    # 1. Load all data sources
    docs_en = load_english_pdfs(PATH_EN_PDFS)
    docs_ta = load_tamil_ocr_pdfs(PATH_TA_PDFS)
    docs_csv = load_csv_files(PATH_CSV)
    
    all_docs = docs_en + docs_ta + docs_csv
    if not all_docs:
        print("No documents found. Exiting.")
        return

    # 2. Clean and Split
    cleaned_docs = clean_document_content(all_docs)
    texts_chunk = split_docs(cleaned_docs)

    # 3. Load Multilingual Embedding Model
    print(f"Loading multilingual embedding model: {EMBEDDING_MODEL}...")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'} # Use 'cuda' if GPU is available, else 'cpu'
    )
    print("Embedding model loaded.")


   # 4. Initialize Pinecone
    pc = Pinecone(api_key=PINECONE_API_KEY_TA)
    
    # DELETE OLD INDEX if it exists, as it used a different model
    if INDEX_NAME in pc.list_indexes().names():
        print(f"Deleting existing index '{INDEX_NAME}'...")
        pc.delete_index(INDEX_NAME)
        time.sleep(5) # Give it a moment

    # 5. Create new index
    print(f"Creating new index '{INDEX_NAME}'...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws", 
            region="us-east-1"
        )
    )
    
    # Wait for index to be ready
    while not pc.describe_index(INDEX_NAME).status['ready']:
        print("Waiting for index to initialize...")
        time.sleep(5)

    print("Index is ready.")

    # 6. Upsert data to Pinecone
    print(f"Upserting {len(texts_chunk)} chunks to Pinecone...")
    PineconeVectorStore.from_documents(
        documents=texts_chunk,
        embedding=embeddings,
        index_name=INDEX_NAME
    )
    
    print("--- Ingestion Complete ---")

if __name__ == "__main__":
    main()

Loading English PDFs from D:\CyChat\CyChat2\data\d-en...


100%|██████████| 16/16 [02:14<00:00,  8.42s/it]


Loading Tamil (OCR) PDFs from D:\CyChat\CyChat2\data\d-ta...
  Processing hithawathi.pdf...
  Processing safetytamil.pdf...
Loaded 107 pages from Tamil PDFs.
Loading CSVs from D:\CyChat\CyChat2\data\csv...


100%|██████████| 1/1 [00:00<00:00,  6.98it/s]


Cleaning all documents...
Splitting documents...
Total chunks created: 6551
Loading multilingual embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_9140\1466540890.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import os
from typing import Dict, Any

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda, RunnableBranch
from langchain_openai import ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain.embeddings import HuggingFaceEmbeddings
from pinecone import Pinecone

# Import translation pipeline
from transformers import pipeline

# --- 1. Load Environment & Models ---
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# --- Configuration ---
INDEX_NAME = "cychat-multilingual"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# --- 2. Initialize Components ---

print("Loading components...")

# LLM for synthesis and language detection
llm = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY)

# Multilingual embedding model (must be the same as ingestion)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'} # Use 'cuda' if GPU is available
)

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embeddings
)

# **Key Change:** Create TWO retrievers with metadata filtering
# Retriever for English documents
retriever_en = vectorstore.as_retriever(
    search_kwargs={'filter': {'lang': 'en'}, 'k': 2}
)
# Retriever for Tamil documents
retriever_ta = vectorstore.as_retriever(
    search_kwargs={'filter': {'lang': 'ta'}, 'k': 2}
)

# Load IndicTrans2 translation model (Tamil to English)
print("Loading IndicTrans2 translator (ta-en)...")
# This model name might change, check ai4bharat on Hugging Face
translator_ta_en = pipeline(
    "translation",
    model="ai4bharat/indictrans2-ta-en", # Using the base model
    torch_dtype="auto", # Use bfloat16 for speed if available
    device=0 if 'cuda' in embeddings.model_kwargs.get('device', 'cpu') else -1
)
print("Translator loaded.")

# --- 3. Define Custom Chain Functions (LCEL) ---

def detect_language(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """Detects language of input and passes all data through."""
    query = inputs["input"]
    prompt = ChatPromptTemplate.from_template(
        "Detect the language of the following text. Respond with only the "
        "two-letter ISO 639-1 code (e.g., 'en' for English, 'ta' for Tamil). "
        "Text: \"{query}\""
    )
    lang_chain = prompt | llm | StrOutputParser()
    lang_code = lang_chain.invoke({"query": query}).strip().lower()
    
    # Default to 'en' if detection is unclear
    if lang_code not in ['en', 'ta']:
        lang_code = 'en'
        
    inputs["lang"] = lang_code
    return inputs

def translate_query(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """Translates query if Tamil, otherwise sets queries to be the same."""
    if inputs["lang"] == 'ta':
        original_query = inputs["input"]
        # IndicTrans2 needs a list of strings
        translated_text = translator_ta_en([original_query], src_lang="tam_Taml", tgt_lang="eng_Latn")
        inputs["en_query"] = translated_text[0]["translation"]
        inputs["ta_query"] = original_query
    else:
        # If input is English, use it for both retrievers
        inputs["en_query"] = inputs["input"]
        inputs["ta_query"] = inputs["input"] # Search Tamil docs with English query
    return inputs

def retrieve_context(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """Retrieves context from both EN and TA retrievers."""
    en_docs = retriever_en.invoke(inputs["en_query"])
    ta_docs = retriever_ta.invoke(inputs["ta_query"])
    
    # Combine and pass along
    inputs["context"] = en_docs + ta_docs
    return inputs

# Combine context documents into a single string
def format_context(inputs: Dict[str, Any]) -> Dict[str, Any]:
    """Formats the retrieved docs into a string."""
    context_str = "\n\n---\n\n".join([doc.page_content for doc in inputs["context"]])
    inputs["context_str"] = context_str
    return inputs

# --- 4. Define Prompts for Synthesis ---

# Prompt for English answers
prompt_en = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful assistant. Answer the user's question based on "
        "the provided context. Answer in English.\n\n"
        "Context:\n{context_str}"
    )),
    ("human", "{input}")
])

# Prompt for Tamil answers
prompt_ta = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful assistant. Answer the user's question based on "
        "the provided context. **You must answer in Tamil.**\n\n"
        "Context:\n{context_str}"
    )),
    ("human", "{input}")
])

# Use RunnableBranch to select the prompt based on detected language
prompt_branch = RunnableBranch(
    (lambda x: x.get("lang") == "ta", prompt_ta), # If lang is 'ta', use Tamil prompt
    prompt_en  # Otherwise, use English prompt
)

# --- 5. Build the Full RAG Chain ---

print("Assembling final RAG chain...")

# This chain is a series of steps, each passing its output to the next
full_rag_chain = (
    RunnablePassthrough()                 # 1. Start with original input ({"input": "..."})
    | RunnableLambda(detect_language)     # 2. Add 'lang' key: ({"input": "...", "lang": "ta"})
    | RunnableLambda(translate_query)     # 3. Add query keys: ({"input": "...", "lang": "ta", "en_query": "...", "ta_query": "..."})
    | RunnableLambda(retrieve_context)    # 4. Add context docs: ({"input": "...", ..., "context": [Doc1, Doc2]})
    | RunnableLambda(format_context)      # 5. Add formatted context: ({"input": "...", ..., "context_str": "..."})
    | prompt_branch                       # 6. Select the correct prompt (EN or TA)
    | llm                                 # 7. Send prompt to LLM
    | StrOutputParser()                   # 8. Get the final string answer
)

print("Chain ready. You can now ask questions.")

# --- 6. Run the Chain ---

# Example 1: English Query
response_en = full_rag_chain.invoke({"input": "What is Phishing?"})
print("\n--- English Query ---")
print(f"Q: What is Phishing?")
print(f"A: {response_en}")

# Example 2: Tamil Query
# (Using a placeholder for a Tamil question)
tamil_question = "சைபர் பாதுகாப்பு என்றால் என்ன?" # "What is cyber security?" in Tamil
response_ta = full_rag_chain.invoke({"input": tamil_question})
print("\n--- Tamil Query ---")
print(f"Q: {tamil_question}")
print(f"A: {response_ta}")